# Subject Reframe

Track one person through a multi-scene video and produce a smooth **9:16, 1080 × 1920** Reel.

The notebook first detects scene boundaries. It then tracks exactly one scene, waits for you to confirm the target, and only afterward processes the next scene. Tracker state is reset at every scene cut.

## 1. Configuration

Edit this section before running the rest of the notebook. Settings are grouped by purpose. Development defaults recompute tracking and ask for every scene selection.

In [ ]:
from pathlib import Path

# ── Project and files ──────────────────────────────────────────────────────
# The notebook works both from the project root and from project/notebooks/.
WORKING_DIRECTORY = Path.cwd().resolve()
PROJECT_ROOT = (
    WORKING_DIRECTORY.parent
    if WORKING_DIRECTORY.name == "notebooks"
    else WORKING_DIRECTORY
)
# Source video to analyze.
INPUT_VIDEO = PROJECT_ROOT / "input" / "source03.mp4"
# First source-video timestamp to process, in seconds.
PROCESS_START_SECONDS = 1.0
# Last source-video timestamp to process; -1 means through the video end.
PROCESS_END_SECONDS = 6.0
# Directory for videos, reports, caches, and saved scene selections.
OUTPUT_DIRECTORY = PROJECT_ROOT / "output"
SILENT_VIDEO = OUTPUT_DIRECTORY / "subject_reel_silent.mp4"
FINAL_VIDEO = OUTPUT_DIRECTORY / "subject_reel.mp4"
BROWSER_PREVIEW_VIDEO = OUTPUT_DIRECTORY / "subject_reel_browser_preview.mp4"
PREVIEW_IMAGE = OUTPUT_DIRECTORY / "preview.jpg"
TRACK_CACHE_DIRECTORY = OUTPUT_DIRECTORY / "cache"
SCENE_SELECTIONS_FILE = OUTPUT_DIRECTORY / "scene_selections.json"

# ── Development and resume behavior ───────────────────────────────────────
# False reuses a cache when the video, detector version, and settings match.
FORCE_SCENE_REDETECTION = True
# True recomputes YOLO tracking and overwrites each scene cache.
FORCE_RETRACK = True
# True asks for confirmation in every scene instead of accepting saved IDs.
REVIEW_SAVED_SELECTIONS = True

# ── Scene detection ────────────────────────────────────────────────────────
# True combines adaptive, hard-cut, and fade detectors; False uses only manual cuts.
AUTOMATIC_SCENE_DETECTION = True
# Extra known cut times in seconds, for example [12.4, 28.1].
MANUAL_SCENE_CUTS = []
# Ignore automatic boundaries closer than this; 0.5 keeps short shots.
MINIMUM_SCENE_SECONDS = 0.5
# Motion-resistant detector: lower values find more subtle scene changes. (2.2)
ADAPTIVE_SCENE_THRESHOLD = 5
# Minimum HSV change for an adaptive cut; lower values increase sensitivity. (10)
ADAPTIVE_MIN_CONTENT_VALUE = 7
# Frames before and after each frame used for the rolling average.
ADAPTIVE_WINDOW_WIDTH = 2
# Also detect ordinary hard cuts from direct colour/intensity changes.
ENABLE_HARD_CUT_DETECTOR = True
# Hard-cut sensitivity: lower values detect more cuts but add false positives. (24)
HARD_CUT_THRESHOLD = 15
# Also detect fades to or from dark frames.
ENABLE_FADE_DETECTOR = True
# Frames darker than this level participate in fade detection.
FADE_BRIGHTNESS_THRESHOLD = 12.0
# Frames skipped between checks. Keep 0 for reliable detection.
SCENE_FRAME_SKIP = 0
# Scene midpoint images shown per review page and row.
SCENES_PER_PREVIEW_PAGE = 9
SCENE_PREVIEW_COLUMNS = 3

# ── Person tracking ────────────────────────────────────────────────────────
# Small YOLO model with a good speed/accuracy balance.
MODEL_NAME = "yolo11s.pt"
# Custom BoT-SORT uses appearance ReID and a longer buffer through occlusions.
# It is slower than the stock tracker but greatly reduces fragmented Track IDs.
TRACKER_NAME = str(PROJECT_ROOT / "config" / "subject_botsort.yaml")
# 16 uses FP16 on CUDA; use None or 32 for FP32.
TRACKING_QUANTIZE = 16
# YOLO inference size. Raise to 640 for small/distant people. (512)
TRACKING_IMAGE_SIZE = 576
# Analyze every Nth source frame. Use 1 for maximum identity stability.
TRACKING_FRAME_STRIDE = 2
# Minimum person-detection confidence.
DETECTION_CONFIDENCE = 0.20
# Non-maximum-suppression overlap threshold.
TRACKING_IOU = 0.50

# ── Per-scene target review ────────────────────────────────────────────────
# Optional representative-frame overrides using absolute source timestamps.
SELECTION_FRAME_OVERRIDES = {}
# Width of each card in the horizontally scrollable track strip.
TRACK_PREVIEW_CARD_WIDTH = 240
# Display height of each same-frame track image.
TRACK_PREVIEW_CARD_HEIGHT = 300
# True permits comma-separated IDs when one person has fragmented tracks.
ALLOW_MULTIPLE_TRACK_SELECTION = True

# ── 9:16 output crop ───────────────────────────────────────────────────────
# "fast": 540×960, reflected edges, linear resize; "high": final quality.
RENDER_QUALITY = "fast"
# Lightweight dimensions used only when RENDER_QUALITY is "fast".
DRAFT_OUTPUT_WIDTH = 540
DRAFT_OUTPUT_HEIGHT = 960
# Full-resolution dimensions used when RENDER_QUALITY is "high".
OUTPUT_WIDTH = 1080
OUTPUT_HEIGHT = 1920
# Width of the lightweight video embedded in the final notebook cell.
BROWSER_PREVIEW_WIDTH = 360
# Extra horizontal room, headroom, and foot room around the detected body.
HORIZONTAL_MARGIN = 0.25
TOP_MARGIN = 0.18
BOTTOM_MARGIN = 0.22
# Virtual-camera smoothing and maximum interpolated tracking gap.
SMOOTHING_SECONDS = 0.45
MAX_INTERPOLATION_SECONDS = 0.60
# Warn when the chosen subject's confidence falls below this value.
LOW_CONFIDENCE_THRESHOLD = 0.35
# High-quality edge fill; fast mode uses the current frame's average colour.
PADDING_MODE = "blur"
# Missing Track ID fallback: "hold" keeps the last reliable crop through occlusion.
# Alternatives: "fit" shows the full frame, "center" centers it, or "black".
MISSING_TRACK_POLICY = "hold"
# Scene selected as absent: "fit" shows the full frame; also supports "center" or "black".
ABSENT_SCENE_POLICY = "fit"

OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

TRACK_CACHE_DIRECTORY.mkdir(parents=True, exist_ok=True)print("Final video:", FINAL_VIDEO)

print("Project root:", PROJECT_ROOT)print("Input video:", INPUT_VIDEO)

## 2. Load the project

Load the reusable package after configuration. Run this whenever source files change.

In [ ]:
import sys

SRC_DIRECTORY = PROJECT_ROOT / "src"
if str(SRC_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SRC_DIRECTORY))

from subject_reframe import (
    CropConfig, PersonTrackingSession, Scene, SKIP_SCENE, build_crop_plan,
    build_tracking_cache_metadata, create_browser_preview, detect_scenes,
    ffmpeg_available,
    inspect_video, load_scene_selections, load_tracking_records,
    mux_original_audio, print_scene_table, print_warning_summary,
    render_video, save_preview, save_report,
    save_scene_selections, save_tracking_records,
    source_segments_from_plans,
    resolve_processing_range, show_scene_midpoints,
    show_scene_track_overview,
    validate_target_tracks,
)
print("Subject Reframe package loaded from:", SRC_DIRECTORY)

## 3. Check Python, FFmpeg, and the RTX GPU

In [ ]:
import cv2
import torch
import ultralytics

# Derived runtime values: use the first CUDA GPU and FP16 when available.
CUDA_AVAILABLE = torch.cuda.is_available()
DEVICE = 0 if CUDA_AVAILABLE else "cpu"
EFFECTIVE_TRACKING_QUANTIZE = TRACKING_QUANTIZE if CUDA_AVAILABLE else None
print("OpenCV:", cv2.__version__)
print("Ultralytics:", ultralytics.__version__)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA runtime:", torch.version.cuda)
print("CUDA available:", CUDA_AVAILABLE)
print("Tracking quantization:", EFFECTIVE_TRACKING_QUANTIZE)
if CUDA_AVAILABLE:
    properties = torch.cuda.get_device_properties(0)
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory (GB):", round(properties.total_memory / 1024**3, 2))
else:
    print("WARNING: CUDA is unavailable; tracking will run on the CPU.")
print("FFmpeg available:", ffmpeg_available())

## 4. Inspect the input and processing range

In [ ]:
if not INPUT_VIDEO.exists():
    raise FileNotFoundError(f"Place your video at {INPUT_VIDEO}, or change INPUT_VIDEO above.")
video_info = inspect_video(INPUT_VIDEO)
processing_range = resolve_processing_range(
    video_info, PROCESS_START_SECONDS, PROCESS_END_SECONDS
)
print(f"Resolution: {video_info.width} × {video_info.height}")
print(f"Frame rate: {video_info.fps:.3f} fps")
print(f"Frames: {video_info.frame_count:,}")
print(f"Duration: {video_info.duration_seconds:.2f} seconds")
print(
    f"Processing: {processing_range.start_seconds:.3f}s–"
    f"{processing_range.end_seconds:.3f}s "
    f"({processing_range.frame_count:,} frames, "
    f"{processing_range.duration_seconds:.3f}s output)"
)

## 5. Detect and review scenes

Review the table and midpoint images. Detection combines a motion-resistant adaptive detector, a hard-cut detector, and a fade detector. Nearby automatic boundaries are merged, while `MANUAL_SCENE_CUTS` remain exact. Raise thresholds if normal movement creates false cuts; lower them when real transitions are missed.

In [ ]:
scene_cache = OUTPUT_DIRECTORY / "scene_cache.json"
if AUTOMATIC_SCENE_DETECTION:
    scenes = detect_scenes(
        INPUT_VIDEO,
        adaptive_threshold=ADAPTIVE_SCENE_THRESHOLD,
        min_content_value=ADAPTIVE_MIN_CONTENT_VALUE,
        window_width=ADAPTIVE_WINDOW_WIDTH,
        use_content_detector=ENABLE_HARD_CUT_DETECTOR,
        content_threshold=HARD_CUT_THRESHOLD,
        use_fade_detector=ENABLE_FADE_DETECTOR,
        fade_threshold=FADE_BRIGHTNESS_THRESHOLD,
        min_scene_seconds=MINIMUM_SCENE_SECONDS,
        frame_skip=SCENE_FRAME_SKIP,
        manual_cut_seconds=MANUAL_SCENE_CUTS,
        start_frame=processing_range.start_frame,
        end_frame=processing_range.end_frame,
        cache_path=scene_cache,
        force_recompute=FORCE_SCENE_REDETECTION,
    )
else:
    manual_cut_frames = {
        int(round(seconds * video_info.fps))
        for seconds in MANUAL_SCENE_CUTS
    }
    cut_frames = sorted({
        processing_range.start_frame, processing_range.end_frame,
        *(
            frame for frame in manual_cut_frames
            if processing_range.start_frame < frame
            < processing_range.end_frame
        ),
    })
    scenes = [
        Scene(i, start, end, video_info.fps)
        for i, (start, end) in enumerate(zip(cut_frames[:-1], cut_frames[1:]))
        if end > start
    ]
print_scene_table(scenes)
show_scene_midpoints(
    INPUT_VIDEO, scenes, scenes_per_page=SCENES_PER_PREVIEW_PAGE,
    columns=SCENE_PREVIEW_COLUMNS,
)

## 6. Track and confirm one scene at a time

This is a synchronous, deterministic workflow: track one scene, show one full-frame preview with green Track ID boxes and individual crops made from that exact same frame, wait for a valid response, save it, and then start the next scene. Enter `S` to remove the entire scene from both final video and audio; `A` keeps the scene when the target is absent.

In [ ]:
tracking_cache_metadata = build_tracking_cache_metadata(
    video_info, scenes, model_name=MODEL_NAME, tracker_name=TRACKER_NAME,
    quantize=EFFECTIVE_TRACKING_QUANTIZE, image_size=TRACKING_IMAGE_SIZE,
    frame_stride=TRACKING_FRAME_STRIDE, confidence=DETECTION_CONFIDENCE,
    iou=TRACKING_IOU,
)

TARGET_TRACK_BY_SCENE = {}
if SCENE_SELECTIONS_FILE.exists() and not FORCE_RETRACK:
    try:
        TARGET_TRACK_BY_SCENE = load_scene_selections(
            SCENE_SELECTIONS_FILE, expected_metadata=tracking_cache_metadata
        )
        print(f"Loaded {len(TARGET_TRACK_BY_SCENE)} confirmed scene selection(s).")
    except (ValueError, KeyError, TypeError) as error:
        print("Ignoring stale scene selections:", error)

if "tracking_session" in globals() and tracking_session is not None:
    tracking_session.close()
tracking_session = None
tracking_records = {}


def get_tracking_session():
    global tracking_session
    if tracking_session is None:
        tracking_session = PersonTrackingSession(
            model_name=MODEL_NAME,
            tracker_name=TRACKER_NAME,
            device=DEVICE,
            quantize=EFFECTIVE_TRACKING_QUANTIZE,
            image_size=TRACKING_IMAGE_SIZE,
            frame_stride=TRACKING_FRAME_STRIDE,
            confidence=DETECTION_CONFIDENCE,
            iou=TRACKING_IOU,
        )
    return tracking_session


def track_scene_with_progress_bar(scene, scene_position):
    return get_tracking_session().track_scene(
        INPUT_VIDEO,
        scene,
        progress=True,
        progress_description=f"Tracking {scene_position}/{len(scenes)}",
    )


def parse_track_ids(answer):
    try:
        selected_ids = sorted({int(value.strip()) for value in answer.split(",")})
    except ValueError:
        return None
    if not selected_ids:
        return None
    if not ALLOW_MULTIPLE_TRACK_SELECTION and len(selected_ids) > 1:
        return None
    return selected_ids


try:
    for scene_position, scene in enumerate(scenes, start=1):
        print(
            f"\nScene {scene_position}/{len(scenes)} · source "
            f"{scene.start_seconds:.2f}s–{scene.end_seconds:.2f}s "
            f"· {scene.frame_count:,} frames"
        )
        scene_cache = TRACK_CACHE_DIRECTORY / f"scene_{scene.index:03d}_tracking.json"
        scene_metadata = build_tracking_cache_metadata(
            video_info,
            [scene],
            model_name=MODEL_NAME,
            tracker_name=TRACKER_NAME,
            quantize=EFFECTIVE_TRACKING_QUANTIZE,
            image_size=TRACKING_IMAGE_SIZE,
            frame_stride=TRACKING_FRAME_STRIDE,
            confidence=DETECTION_CONFIDENCE,
            iou=TRACKING_IOU,
        )

        scene_records = None
        if scene_cache.exists() and not FORCE_RETRACK:
            try:
                scene_records = load_tracking_records(
                    scene_cache, expected_metadata=scene_metadata
                )
                print(f"Loaded tracking cache for scene {scene.index}.")
            except (ValueError, KeyError, TypeError) as error:
                print(f"Ignoring stale scene {scene.index} cache: {error}")

        if scene_records is None:
            scene_records = track_scene_with_progress_bar(
                scene, scene_position
            )
            save_tracking_records(scene_records, scene_cache, metadata=scene_metadata)
            print(f"Saved tracking cache for scene {scene.index}.")
        tracking_records.update(scene_records)

        saved_selection = scene.index in TARGET_TRACK_BY_SCENE
        if saved_selection and not REVIEW_SAVED_SELECTIONS and not FORCE_RETRACK:
            problems = validate_target_tracks(
                [scene],
                scene_records,
                {scene.index: TARGET_TRACK_BY_SCENE[scene.index]},
            )
            if not problems:
                print(
                    f"Using saved scene {scene.index} selection: "
                    f"{TARGET_TRACK_BY_SCENE[scene.index]}"
                )
                continue
            print("Saved selection is no longer valid; choose again.")

        reviewed_track_ids = set()
        while True:
            preview_frame, visible_track_ids = show_scene_track_overview(
                INPUT_VIDEO,
                scene,
                scene_records,
                override_seconds=SELECTION_FRAME_OVERRIDES,
                card_width=TRACK_PREVIEW_CARD_WIDTH,
                card_image_height=TRACK_PREVIEW_CARD_HEIGHT,
            )
            visible_track_ids = set(visible_track_ids)
            reviewed_track_ids.update(visible_track_ids)
            retrack_requested = False
            refresh_preview = False

            while True:
                commands = ["visible Track ID"]
                if ALLOW_MULTIPLE_TRACK_SELECTION:
                    commands.append("comma-separated reviewed IDs")
                commands.extend([
                    "F = choose another preview time",
                    "A = target absent",
                    "S = skip scene from final video",
                    "R = retrack scene",
                ])
                print("Options: " + " | ".join(commands))
                answer = input(
                    f"Scene {scene_position}/{len(scenes)} selection: "
                ).strip()
                command = answer.lower()

                if command in {"f", "frame", "time"}:
                    requested = input(
                        f"Absolute source time for this scene "
                        f"({scene.start_seconds:.2f} <= time < "
                        f"{scene.end_seconds:.2f}): "
                    ).strip()
                    try:
                        requested_seconds = float(requested)
                    except ValueError:
                        print("Invalid timestamp. Enter seconds as a number.")
                        continue
                    if not (
                        scene.start_seconds <= requested_seconds
                        < scene.end_seconds
                    ):
                        print("Timestamp is outside this scene.")
                        continue
                    SELECTION_FRAME_OVERRIDES[scene.index] = requested_seconds
                    refresh_preview = True
                    break

                if command in {"r", "retrack"}:
                    retrack_requested = True
                    break

                if command in {"s", "skip"}:
                    selection = SKIP_SCENE
                elif command in {"a", "absent", "none"}:
                    selection = None
                else:
                    selected_ids = parse_track_ids(answer)
                    if selected_ids is None:
                        print("Invalid response. Enter visible integer IDs, F, A, S, or R.")
                        continue
                    hidden_ids = set(selected_ids) - reviewed_track_ids
                    if hidden_ids:
                        print(
                            "These IDs have not appeared in a reviewed frame: "
                            + ", ".join(map(str, sorted(hidden_ids)))
                            + ". Enter F to display another frame."
                        )
                        continue
                    selection = (
                        selected_ids[0]
                        if len(selected_ids) == 1
                        else selected_ids
                    )

                problems = validate_target_tracks(
                    [scene], scene_records, {scene.index: selection}
                )
                if problems:
                    print("Invalid selection: " + "; ".join(problems))
                    continue
                break

            if refresh_preview:
                continue

            if retrack_requested:
                print(f"Retracking scene {scene.index} as requested.")
                TARGET_TRACK_BY_SCENE.pop(scene.index, None)
                scene_records = track_scene_with_progress_bar(
                    scene, scene_position
                )
                save_tracking_records(
                    scene_records, scene_cache, metadata=scene_metadata
                )
                tracking_records.update(scene_records)
                reviewed_track_ids.clear()
                continue

            TARGET_TRACK_BY_SCENE[scene.index] = selection
            save_scene_selections(
                TARGET_TRACK_BY_SCENE,
                SCENE_SELECTIONS_FILE,
                metadata=tracking_cache_metadata,
            )
            print(f"Confirmed scene {scene.index}: {selection}")
            break

        if scene_position < len(scenes):
            print(f"Starting scene {scenes[scene_position].index} next…", flush=True)
finally:
    if tracking_session is not None:
        tracking_session.close()
        tracking_session = None

print(f"\nCompleted {len(TARGET_TRACK_BY_SCENE)} of {len(scenes)} scene selections.")



## 7. Selection summary

Section 6 finishes only after every scene has a valid selection, so this summary is safe to run immediately afterward.

In [ ]:
for scene in scenes:
    print(
        f"Scene {scene.index:>3}  {scene.start_seconds:>7.2f}s–"
        f"{scene.end_seconds:>7.2f}s  selection={TARGET_TRACK_BY_SCENE.get(scene.index)}"
    )


## 8. Validate all selections

This prevents crop planning when any scene is unfinished or references a Track ID that is absent from its scene cache.

In [ ]:
unfinished = [scene.index for scene in scenes if scene.index not in TARGET_TRACK_BY_SCENE]
if unfinished:
    raise ValueError(f"Unfinished scene selections: {unfinished}")
selection_problems = validate_target_tracks(scenes, tracking_records, TARGET_TRACK_BY_SCENE)
if selection_problems:
    raise ValueError("Selection problems:\n- " + "\n- ".join(selection_problems))
print("All scene selections are valid.")

## 9. Plan and inspect the 9:16 crop

Short tracking gaps are interpolated. Every available padded body box is guaranteed to remain inside its crop even after camera smoothing. Longer occlusions hold the nearest reliable subject crop instead of jumping to the full frame. Subject crops remain vertically centered; fast rendering uses an average-colour edge fill and high-quality rendering uses blurred edge fill when the portrait crop extends beyond the source.

In [ ]:
if RENDER_QUALITY not in {"fast", "high"}:
    raise ValueError("RENDER_QUALITY must be 'fast' or 'high'")
render_width = DRAFT_OUTPUT_WIDTH if RENDER_QUALITY == "fast" else OUTPUT_WIDTH
render_height = DRAFT_OUTPUT_HEIGHT if RENDER_QUALITY == "fast" else OUTPUT_HEIGHT
render_padding_mode = "average" if RENDER_QUALITY == "fast" else PADDING_MODE
crop_config = CropConfig(
    output_width=render_width, output_height=render_height,
    horizontal_margin=HORIZONTAL_MARGIN, top_margin=TOP_MARGIN,
    bottom_margin=BOTTOM_MARGIN, smoothing_seconds=SMOOTHING_SECONDS,
    max_interpolation_seconds=MAX_INTERPOLATION_SECONDS,
    low_confidence_threshold=LOW_CONFIDENCE_THRESHOLD,
    padding_mode=render_padding_mode, render_quality=RENDER_QUALITY,
    missing_track_policy=MISSING_TRACK_POLICY,
    absent_scene_policy=ABSENT_SCENE_POLICY,
)
crop_plans, warnings = build_crop_plan(
    video_info, scenes, tracking_records, TARGET_TRACK_BY_SCENE, crop_config
)
print(f"Render mode: {RENDER_QUALITY} · {render_width}×{render_height} · {render_padding_mode} padding")
print_warning_summary(warnings)

## 10. Preview one planned frame

In [ ]:
from IPython.display import Image, display
preview_path = save_preview(INPUT_VIDEO, crop_plans, video_info, crop_config, PREVIEW_IMAGE)
if preview_path is None:
    print("No preview: every scene is marked absent.")
else:
    display(Image(filename=str(preview_path), width=360))

## 11. Render and restore audio

Frames are cropped and sent directly to FFmpeg for browser-compatible H.264 encoding. NVENC uses the RTX GPU when available, with an automatic CPU fallback. The matching source audio is then attached without encoding the video a second time.

In [ ]:
rendered_path = render_video(
    INPUT_VIDEO, SILENT_VIDEO, crop_plans, video_info, crop_config
)
print("Rendered silent video:", rendered_path)
source_audio_segments = source_segments_from_plans(crop_plans, video_info.fps)
print(f"Retained source segments: {len(source_audio_segments)}")
if ffmpeg_available():
    final_path = mux_original_audio(
        SILENT_VIDEO, INPUT_VIDEO, FINAL_VIDEO,
        start_seconds=processing_range.start_seconds,
        duration_seconds=processing_range.duration_seconds,
        copy_video=True,
        source_segments=source_audio_segments,
    )
else:
    final_path = SILENT_VIDEO
    print("WARNING: FFmpeg is unavailable, so the result has no audio.")
print("Final video:", final_path)

## 12. Save reports and play the result

In [ ]:
from IPython.display import FileLink, Video, display
report_path, warnings_path = save_report(
    OUTPUT_DIRECTORY, info=video_info, scenes=scenes,
    config=crop_config, warnings=warnings,
    tracking_metadata=tracking_cache_metadata,
    processing_range={
        "start_frame": processing_range.start_frame,
        "end_frame": processing_range.end_frame,
        "start_seconds": processing_range.start_seconds,
        "end_seconds": processing_range.end_seconds,
        "duration_seconds": processing_range.duration_seconds,
    },
)
print("Report:", report_path)
print("Warnings:", warnings_path)
final_path = Path(final_path).resolve()
browser_preview_path = create_browser_preview(
    final_path, BROWSER_PREVIEW_VIDEO, width=BROWSER_PREVIEW_WIDTH
)
# embed=True stores the small preview in the cell output, avoiding Jupyter URL issues.
display(Video(
    filename=str(browser_preview_path), embed=True,
    width=BROWSER_PREVIEW_WIDTH, html_attributes="controls playsinline preload='metadata'",
))
display(FileLink(str(final_path), result_html_prefix="Open or download full video: "))

## Limitations

- Body parts outside the source frame cannot be restored.
- A long occlusion can change a Track ID within one scene; review missing-tracking warnings.
- Crops smaller than 1080 × 1920 are still rendered, but the required upscale factor is recorded in `warnings.csv`.
- Tracking every second frame is the balanced default. Use `TRACKING_FRAME_STRIDE = 1` if very fast motion causes identity instability.
- Compatible caches load automatically; set `FORCE_RETRACK = True` only when you deliberately want to recompute identical settings.